# Farmers Guide — Satellite-only Training

Spatial K-fold CV of `SatelliteCNN` on Sentinel-2 cubes paired with RALS-derived maize yields.

Run this on Colab with a **GPU runtime**; everything else is a thin import from `src/farmers_guide/`.

In [ ]:
!pip install -q git+https://github.com/Hus211/FarmersGuide.git

In [ ]:
import torch

import config
from farmers_guide.data.hdf5_builder import build_cube
from farmers_guide.data.dataset import read_labels, synthetic_labels
from farmers_guide.train import train

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected — training will be very slow. Switch to a GPU runtime via Runtime > Change runtime type.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if not config.DRIVE_ROOT.exists():
    raise FileNotFoundError(f"DRIVE_ROOT not found after mount: {config.DRIVE_ROOT}")
print(f"Drive mounted; DRIVE_ROOT={config.DRIVE_ROOT}")

In [ ]:
# TODO: multi-AOI training via ConcatDataset
import h5py

cube_path = build_cube("chilanga", "2018_19")
print(f"cube: {cube_path}")
with h5py.File(cube_path, "r") as f:
    print(f"shape (T, H, W, C): {f['cube'].shape}")
    dates = [d.decode('ascii') for d in f['dates'][:]]
    print(f"dates: {dates[0]} .. {dates[-1]}  (n={len(dates)})")

In [ ]:
if config.GROUND_TRUTH_CSV.exists():
    labels = read_labels(config.GROUND_TRUTH_CSV, aoi="chilanga", season="2018_19")
else:
    print("Ground-truth labels not yet available; falling back to synthetic for pipeline validation")
    labels = synthetic_labels(n=50, aoi="chilanga", season="2018_19")
print(f"labels: {len(labels)} rows")

In [ ]:
results = train(cube_path, labels, n_splits=5, epochs=config.EPOCHS)

In [ ]:
import numpy as np
import pandas as pd

rmses = np.array([r["best_val_rmse_kg_ha"] for r in results])
r2s = np.array([r["best_val_r2"] for r in results])
print(f"val RMSE: {rmses.mean():.1f} ± {rmses.std():.1f} kg/ha  (target < {config.TARGET_RMSE_KG_HA})")
print(f"val R^2 : {r2s.mean():.3f} ± {r2s.std():.3f}       (target > {config.TARGET_R2})")

pd.DataFrame([
    {"fold": r["fold_index"], "best_epoch": r["best_epoch"],
     "val_rmse_kg_ha": r["best_val_rmse_kg_ha"], "val_r2": r["best_val_r2"]}
    for r in results
])

Per-fold checkpoints have been written to `config.WEIGHTS_DIR`. See `src/farmers_guide/evaluate.py` for spatial / temporal hold-out evaluation, baseline comparisons (NDVI-mean OLS, satellite-only, photo-only), and the thesis figures called for in CLAUDE.md §4.